# Dueling / Duelling DQN Teorisi

Bu notebook Dueling DQN fikrini teorik olarak açıklar. Dosya adında kullanıcının yazdığı İngiliz İngilizcesi `duelling` korunmuştur; literatürde yaygın başlık `Dueling DQN` olarak geçer.

Dueling DQN, DQN ailesinin bir mimari varyantıdır; bu yüzden off-policy, replay buffer kullanan ve bootstrap TD target ile eğitilen değer tabanlı bir yöntemdir. Dueling kısmı algoritmanın veri rejimini değil, Q ağının $V(s)$ ve $A(s,a)$ ayrıştırmasını değiştirir.

Odak noktaları:
- DQN'deki değer tahmini problemi
- State value ve advantage ayrımı
- Tanımlanabilirlik problemi ve mean-subtraction çözümü
- Dueling mimarinin neden özellikle çok aksiyonlu ortamlarda faydalı olduğu
- Orbit Wars'taki branch Q başlıklarına uygulanışı

## 1. DQN'deki Temel Sınırlama

Standart DQN tek bir fonksiyon öğrenir:

$$Q(s,a)$$

Bu fonksiyon iki bilgiyi aynı skalar içinde taşır:

- Durumun genel olarak iyi/kötü olması
- Belirli bir aksiyonun o durumda ortalamaya göre ne kadar iyi olması

Bazı durumlarda birçok aksiyonun sonuçları birbirine yakındır. Örneğin Orbit Wars'ta hiçbir iyi saldırı fırsatı yoksa farklı hedeflerin çoğu benzer derecede kötüdür. Böyle durumlarda her aksiyon için ayrı ayrı durum değerini tekrar öğrenmek verimsizdir.

## 2. Value ve Advantage Ayrımı

Dueling DQN şu ayrıştırmayı kullanır:

$$Q(s,a) = V(s) + A(s,a)$$

- $V(s)$: sadece durumun değeri. Bu pozisyonda olmak genel olarak iyi mi?
- $A(s,a)$: aksiyonun göreli avantajı. Bu aksiyon aynı durumdaki diğer aksiyonlardan ne kadar iyi?

Sezgisel olarak:

$$A(s,a) = Q(s,a) - V(s)$$

Eğer durum zaten çok iyi ise birçok aksiyonun Q değeri yüksek olabilir. Dueling mimari bu ortak kısmı $V(s)$ içinde paylaşır, aksiyon başlığı ise farkları öğrenir.

## 3. Tanımlanabilirlik Problemi

$Q(s,a)=V(s)+A(s,a)$ ayrıştırması tekil değildir. Herhangi bir sabit $c$ için:

$$V'(s)=V(s)+c$$

$$A'(s,a)=A(s,a)-c$$

aynı $Q(s,a)$ değerini üretir. Bu, ağın $V$ ve $A$ parçalarını anlamlı şekilde ayırmasını zorlaştırır.

Dueling DQN bu problemi avantajların ortalamasını çıkararak çözer:

$$Q(s,a)=V(s)+A(s,a)-\frac{1}{|\mathcal{A}|}\sum_{a'}A(s,a')$$

Böylece avantajların ortalaması sıfıra merkezlenir. $V(s)$ yaklaşık olarak aksiyonlar üzerindeki ortalama Q seviyesi gibi davranır.

In [ ]:
import torch

def dueling_combine(value, advantage, valid_mask=None):
    """Q = V + A - mean(A) birleşiminin maskeli gösterimi."""
    if valid_mask is None:
        centered_advantage = advantage - advantage.mean(dim=-1, keepdim=True)
    else:
        masked_adv = advantage.masked_fill(~valid_mask, 0.0)
        denom = valid_mask.sum(dim=-1, keepdim=True).clamp(min=1)
        mean_adv = masked_adv.sum(dim=-1, keepdim=True) / denom
        centered_advantage = advantage - mean_adv
    return value.unsqueeze(-1) + centered_advantage

V = torch.tensor([2.0])
A = torch.tensor([[0.2, 1.0, -0.4, 0.0]])
dueling_combine(V, A)

## 3.1. Minimal Dueling DQN Neural Network

Dueling mimaride ağ iki başlık üretir:

- $V(s)$: durumun genel değeri
- $A(s,a)$: aksiyonun o durumdaki göreli avantajı

Sonra bunlar $Q(s,a)=V(s)+A(s,a)-mean_a A(s,a)$ ile birleştirilir. Aşağıdaki ağ, bu ayrıştırmanın minimal PyTorch karşılığıdır.

In [ ]:
try:
    import torch
    import torch.nn as nn
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class MinimalDuelingDQN(nn.Module):
    """Dueling Q ağı: ortak temsil -> value head + advantage head."""
    def __init__(self, obs_dim=16, hidden_dim=128, n_actions=6):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.value_head = nn.Linear(hidden_dim, 1)
        self.advantage_head = nn.Linear(hidden_dim, n_actions)

    def forward(self, obs):
        h = self.encoder(obs)
        value = self.value_head(h)                  # (B, 1)
        advantage = self.advantage_head(h)          # (B, n_actions)
        q_values = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q_values, value.squeeze(1), advantage

net = MinimalDuelingDQN()
obs = torch.randn(3, 16)
q_values, values, advantages = net(obs)

print("V(s) shape:", values.shape)
print("A(s,a) shape:", advantages.shape)
print("Q(s,a) shape:", q_values.shape)
print("Advantage mean after centering:", (q_values - values[:, None]).mean(dim=1))

## 3.2. Dueling DQN Update Algoritması

Dueling DQN'in update algoritması standart DQN ile aynıdır; değişen şey sadece `forward` içinde Q değerlerinin $V(s)$ ve $A(s,a)$ üzerinden üretilmesidir. Bu yüzden TD target, replay buffer, target network ve optimizer adımları değişmez.

In [ ]:
import torch.nn.functional as F

def dueling_dqn_train_step(online_net, target_net, optimizer, obs, actions, rewards, next_obs, dones, gamma=0.99):
    q_values, values, advantages = online_net(obs)
    q_taken = q_values.gather(1, actions[:, None]).squeeze(1)

    with torch.no_grad():
        target_q_values, _, _ = target_net(next_obs)
        td_target = rewards + gamma * (1.0 - dones) * target_q_values.max(dim=1).values

    loss = F.smooth_l1_loss(q_taken, td_target)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(online_net.parameters(), max_norm=10.0)
    optimizer.step()

    return {
        "loss": float(loss.detach()),
        "value_mean": float(values.mean().detach()),
        "advantage_abs_mean": float(advantages.abs().mean().detach()),
    }

online = MinimalDuelingDQN(obs_dim=16, n_actions=6)
target = MinimalDuelingDQN(obs_dim=16, n_actions=6)
target.load_state_dict(online.state_dict())
optimizer = torch.optim.Adam(online.parameters(), lr=1e-3)

stats = dueling_dqn_train_step(
    online,
    target,
    optimizer,
    obs=torch.randn(8, 16),
    actions=torch.randint(0, 6, (8,)),
    rewards=torch.randn(8),
    next_obs=torch.randn(8, 16),
    dones=torch.randint(0, 2, (8,)).float(),
)
stats

## 4. Mimari Yapı

Dueling DQN iki başlıklı bir ağdır:

1. Ortak encoder: durumdan temsil çıkarır.
2. Value head: $V(s)$ üretir.
3. Advantage head: her aksiyon için $A(s,a)$ üretir.
4. Combine layer: $Q(s,a)$ değerlerini oluşturur.

Kayıp fonksiyonu değişmez. Dueling DQN hala DQN TD hedefiyle eğitilir:

$$y = r + \gamma(1-d)Q_{\theta^-}(s', \arg\max_{a'}Q_\theta(s',a'))$$

Yani Dueling DQN bir hedef algoritması değil, Q ağının parametrizasyonudur. Double DQN, replay buffer ve target network ile birlikte kullanılabilir.

## 5. Neden Faydalı?

Dueling mimari özellikle şu durumlarda işe yarar:

- Aksiyon sayısı büyükse
- Birçok aksiyonun değeri birbirine yakınsa
- Bazı durumlarda hangi aksiyonun seçildiği, durumun genel kalitesinden daha az önemliyse
- Maskeli ve faktörize aksiyon uzayı varsa

Örnek: Orbit Wars'ta erken oyunda sahip olunan gezegen sayısı, üretim kapasitesi ve konumsal avantaj durumu genel olarak değerli kılar. Hangi exact hedefe kaç gemi gönderileceği ise bu genel değerin üzerinde daha küçük bir fark yaratabilir. Dueling başlık bu iki sinyali ayırmaya çalışır.

## 6. Orbit Wars'ta Branch Bazlı Dueling

Repo'daki `OrbitWarsQNet` üç Q branch'i üretir:

- `q_src`: kaynak gezegen / NO-OP değeri
- `q_tgt`: kaynak-hedef çifti değeri
- `q_ship`: kaynak-gemi oranı değeri

Toplam değer:

$$Q_{total}(s,src,tgt,ship)=Q_{src}(s,src)+\mathbf{1}_{src\neq NOOP}(Q_{tgt}(s,src,tgt)+Q_{ship}(s,src,ship))$$

Dueling açıkken her branch için ham skorlar advantage gibi yorumlanır ve ortak bir state value eklenir:

$$Q_b(s,a_b)=V(s)+A_b(s,a_b)-mean(A_b(s,\cdot))$$

Kodda `v_head` global token'dan $V(s)$ üretir. Bu global token, Set Transformer encoder'ın tüm gezegen/fleet bilgisiyle etkileşmiş durum özetidir.

In [ ]:
# Repo kökünden çalıştırıldığında Dueling Q ağını oluşturma örneği.
# Bu hücre Orbit Wars bağımlılıkları kuruluysa çalışır.
import sys
from pathlib import Path

repo = Path.cwd()
if (repo / "model_q.py").exists():
    sys.path.insert(0, str(repo))
elif (repo / "Orbit_Wars" / "model_q.py").exists():
    sys.path.insert(0, str(repo / "Orbit_Wars"))

try:
    from model_q import OrbitWarsQNet
    net = OrbitWarsQNet(dueling=True, d_model=128, n_heads=4, n_layers=4)
    print(f"Dueling açık mı? {net.dueling}")
    print(f"Parametre sayısı: {sum(p.numel() for p in net.parameters()):,}")
except Exception as exc:
    print("Orbit Wars modülleri import edilemedi:", exc)

## 7. Maskeler ve Ortalama Alma İnceliği

Teoride mean-subtraction tüm aksiyonlar üzerinde yapılır. Pratikte padding veya geçersiz aksiyonlar varsa soru şudur: ortalama sadece geçerli aksiyonlar üzerinde mi alınmalı?

İdeal maskeli form:

$$Q(s,a)=V(s)+A(s,a)-\frac{1}{|\mathcal{A}_{valid}(s)|}\sum_{a'\in\mathcal{A}_{valid}(s)}A(s,a')$$

Orbit Wars kodunda branch skorları üretildikten sonra seçim aşamasında maskelenir. Bu tasarım basittir ve sabit boyutlu tensorlerle uyumludur. Daha hassas bir varyant, avantaj ortalamasını her durumdaki geçerli kaynak/hedef aksiyonları üzerinden hesaplar; özellikle padding oranı yüksekse bu fark ölçülebilir hale gelebilir.

## 8. Dueling DQN ile DQN Arasındaki Eğitim Farkı

Eğitim döngüsü açısından fark küçüktür:

- Replay buffer aynı kalır.
- Target network aynı kalır.
- Double DQN hedefi aynı kalır.
- TD loss aynı kalır.
- Sadece Q ağının forward parametrizasyonu değişir.

Bu yüzden repo komutu da aynıdır, sadece `--dueling` flag'i eklenir:

```bash
python -m rl.dqn_train --dueling --epochs 100 --games-per-epoch 5
```

Karşılaştırma yaparken aynı seed, aynı replay capacity, aynı epsilon schedule ve aynı reward ayarları kullanılmalıdır. Aksi halde fark mimariden değil eğitim koşullarından gelebilir.

## 9. İzlenecek Metrikler

| Metrik | Ne anlatır? | Dueling yorumu |
| --- | --- | --- |
| `td_error` | Bellman hedefine uzaklık | Daha stabil düşüş beklenebilir |
| `q_mean` | Q ölçeği | Aşırı büyüme divergence belirtisi olabilir |
| `grad_norms` | Hangi modüller sinyal alıyor | `v_head` ve branch head dengesi izlenebilir |
| Win-rate | Oyuncu performansı | Self-play dengesinde yorumlanmalı |
| Aksiyon çeşitliliği | Keşif ve politika çökmesi | Sadece NO-OP veya tek hedefe çökme takip edilmeli |

Dueling DQN her zaman standart DQN'den iyi olacak diye bir garanti yoktur. Avantaj, aksiyon değerlerinin çok farklı olmadığı ve durum değerinin paylaşılabilir olduğu problemler için beklenir.